# Doublet Analysis with second sequencing run. Only doublet scores added. Use Hugo Symbols, not Ensembl ID.

Publication source copy for the Doublets 2 and 4 sequencing analyses. Outputs and execution counts are cleared. See `../VALIDATION_REPORT.md` for the separate execution results and remaining limitations.

Run from the `analysis/` directory after supplying the bundled inputs. Final figure sections explicitly load preserved manuscript checkpoints; they are distinct from fresh clustering.

In [ ]:
from pathlib import Path
Path("../results/Doublets_2_Step1_PreProcessing_halveSingletReads_with_DoubletScores_only_HugoGeneSymbols").mkdir(parents=True, exist_ok=True)

In [ ]:
import pandas as pd
pd.DataFrame.iteritems = pd.DataFrame.items #pandas version incompatibility fix. Reference: https://github.com/YosefLab/Compass/issues/92
import numpy as np
import scanpy as sc
from anndata import read_h5ad
from anndata import read_text
from collections import Counter
import matplotlib.pyplot as plt
from matplotlib.pyplot import figure
from matplotlib.pyplot import rc_context

import anndata
import seaborn as sns
import regex as re
import os

In [ ]:
sc.settings.verbosity = 3
sc.settings.set_figure_params(dpi=200, dpi_save=300)
sc.settings.n_jobs = 30
pd.options.display.max_columns = None
pd.options.display.max_rows = None

sc.logging.print_header()

import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

# Convert Ensembl ID to Hugo Gene Symbols in gene count matrix

In [ ]:
path_data = '../data/Doublets/data/smartseq_doublets_2_analysis/'
df = pd.read_csv(path_data+'genetables.2p/RSM_gRDS_and_stats_table_clean.tsv.gz', sep='\t')
df.head()

# #get gene symbols
df['01_gene_id'] = [i.split('_')[-1] for i in df['01_gene_id']]
df.head()

# Create a boolean mask to identify duplicates in gene symbol
gene_symbol_mask = df['01_gene_id'].duplicated(keep='first')

# Use the mask to filter out the duplicate rows
df_no_duplicates = df[~gene_symbol_mask]
df_no_duplicates.head()

In [ ]:
#save as compressed file
df_no_duplicates.to_csv(path_data+'genetables.2p/RSM_gRDS_and_stats_table_clean_GeneSymbol_NoDuplicates.tsv.gz', compression='gzip', sep='\t', index=False)

# Import Doublet data from tsv.gz

Only need to do this once. Use **h5ad** file after this step.

In [ ]:
path_data = '../data/Doublets/data/smartseq_doublets_2_analysis/'
path_sequencing = '../data/Doublets/sequencing/Novogene_Submissions/'

adata = sc.read_text(path_data+'genetables.2p/RSM_gRDS_and_stats_table_clean_GeneSymbol_NoDuplicates.tsv.gz')
adata

In [ ]:
#Extract metadata from cell name
adata = adata.transpose()
adata.obs['SequencingRun'] = [i.split('_')[0] for i in adata.obs_names] 
adata.obs['SampleID'] = [i.split('_')[1] for i in adata.obs_names] 
adata.obs['SampleType'] = [i.split('_')[2] for i in adata.obs_names]
adata.obs['age'] = [i.split('_')[3] for i in adata.obs_names]
adata.obs['sex'] = [i.split('_')[4] for i in adata.obs_names]
adata.obs['sortedtype'] = [i.split('_')[5] for i in adata.obs_names]
adata.obs['sortedpop'] = [i.split('_')[6] for i in adata.obs_names]
adata.obs['96plate'] = [i.split('_')[7] for i in adata.obs_names]
adata.obs['96plateloc'] = [i.split('_')[8] for i in adata.obs_names]
adata.obs['384plate'] = [i.split('_')[9] for i in adata.obs_names]
adata.obs['384plateloc'] = [i.split('_')[10] for i in adata.obs_names]
adata.obs['barcode'] = [i.split('_')[11] for i in adata.obs_names]
adata.obs['cell'] = adata.obs_names
adata

In [ ]:
# This function allows users to access genes using their gene name
# Import all gene names from adata

gene_list = adata.var_names
def find_genes(*genes, mode='endswith'):
    list = []
    
    # if user passed in a list of genes instead of individual genes
    if type(genes[0]) is not str:
        genes = genes[0]
        
    if mode == 'endswith':
        for g in genes:
            list.extend([i for i in gene_list if i.split()[0].endswith('' + g)])
    if mode == 'exact':
        for g in genes:
            list.extend([i for i in gene_list if i.split('_')[1] == g])
    if mode == 'contains':
        for g in genes:
            list.extend([i for i in gene_list if g in i.split()[0]])    
    print(list)
    return(list)

# Write h5ad file

Do not overwrite!

In [ ]:
# #save tcr data without preprocessing

adata.write_h5ad(path_data+'Doublets_3456cells_GeneSymbols_nopreprocessing.h5ad')

# Quick load adata from h5ad

In [ ]:
path_data = '../data/Doublets/data/smartseq_doublets_2_analysis/'

adata = read_h5ad(path_data+'Doublets_3456cells_GeneSymbols_nopreprocessing.h5ad')

adata

# Quality Control

In [ ]:
sc.pl.highest_expr_genes(adata, n_top=20)

In [ ]:
# Remove ERCC reads
adata.obs['n_countswercc'] = np.sum(adata.X, axis=1)
ercc_gene_mask = [gene.split('_')[0].startswith('ERCC-') for gene in adata.var_names]
adata.obs['ercc_frac'] = adata.X[:, ercc_gene_mask].sum(1)/adata.obs['n_countswercc']

ercc_genes = [name for name in adata.var_names if name.startswith('ERCC-')]
adata = adata[:,~adata.var_names.isin(ercc_genes)]

In [ ]:
adata.obs['n_counts'] = np.sum(adata.X, axis=1)
adata

In [ ]:
# Quality control - calculate QC covariates
adata.obs['n_counts'] = adata.X.sum(1)
adata.obs['n_genes'] = (adata.X > 0).sum(1)
# Mitochondrial genes

mt_gene_mask = [gene.startswith('MT-') for gene in adata.var_names]

adata.obs['mt_frac'] = adata.X[:, mt_gene_mask].sum(1)/adata.obs['n_counts']

In [ ]:
sc.pl.violin(adata, ['n_genes','mt_frac','ercc_frac'],groupby='sortedpop',jitter=0.4,stripplot=True,multi_panel=True,rotation=90)

In [ ]:
sc.pl.scatter(adata,x='FOXP3',y='mt_frac',color='sortedpop')
sc.pl.scatter(adata,x='GZMB',y='mt_frac',color='sortedpop')
sc.pl.scatter(adata,x='FASLG',y='mt_frac',color='sortedpop')

In [ ]:
sc.pl.violin(adata, ['n_genes', 'n_counts', 'mt_frac'], jitter=0.4, multi_panel=True)
sc.pl.scatter(adata, x='n_counts', y='mt_frac')
sc.pl.scatter(adata, x='n_counts', y='n_genes')
sc.pl.scatter(adata, x='n_countswercc', y='ercc_frac')

In [ ]:
#filter cells for gene counts
print(len(adata))
adata = adata[adata.obs['n_counts']>=2000]
print(len(adata))
sc.pp.filter_cells(adata, min_genes=500)
sc.pp.filter_genes(adata, min_cells=3)

In [ ]:
#filter cells for mtrna and ercc
adata = adata[adata.obs.mt_frac < 0.5, :]
adata = adata[adata.obs.ercc_frac < 0.3, :]
adata

# Pre-Processing - halve singlet normalized counts

In [ ]:
adata_singlet = adata[adata.obs['sortedtype']=='Singlet']
adata_singlet
adata_doublet = adata[(adata.obs['sortedtype']=='Doublet-Artificial') | (adata.obs['sortedtype']=='Doublet-Real')]
adata_doublet

In [ ]:
sc.pp.normalize_total(adata_singlet, target_sum=1e4)
sc.pp.normalize_total(adata_doublet, target_sum=1e4)

In [ ]:
# #double counts for doublets

# adata_doublet.X

#halve counts for singlets
adata_singlet.X = 0.5*adata_singlet.X
adata_singlet.X

In [ ]:
adata = adata_doublet.concatenate(adata_singlet)
sc.pp.log1p(adata)

In [ ]:
adata.raw = adata

In [ ]:
# Batch correct by sequencing run
sc.pp.combat(adata, key='SequencingRun', covariates=None, inplace=True)

In [ ]:
adata.uns['log1p']["base"] = None
sc.pp.highly_variable_genes(adata, min_mean=0.0125, max_mean=3, min_disp=0.5)
sc.pl.highly_variable_genes(adata)

In [ ]:
adata.var.head()

# Scale

In [ ]:
sc.pp.scale(adata, max_value=10)

# Save processed data

Do not overwrite!

In [ ]:
#save file

adata.write_h5ad(path_data+'Doublets_3363cells_GeneSymbols_preprocessed_2xDoubletCounts.h5ad')

# Add Doublet quality scores

Rubric:  

0. Singlet/triplet/not touching
1. No clustering of ICAM/LFA1
2. Intermediate clustering
3. Diffuse ICAM-1 or LFA-1 staining at the synapse
4. ICAM OR LFA1 tightly cluster
5. ICAM AND LFA1 tightly cluster

In [ ]:
def convert_96plateloc_to_96plate(str_96plateloc_p):
    int_96plate = int(str_96plateloc_p)
    if int_96plate <= 12:
        return str(int_96plate).zfill(2)
    elif int_96plate > 12 and int_96plate <= 24:
        return str(int_96plate % 13 + 1).zfill(2)
    elif int_96plate > 24 and int_96plate <= 36:
        return str(int_96plate % 25 + 1).zfill(2)

convert_96plateloc_to_96plate('25')

In [ ]:
# Get doublet grades and concatenate them. Also, add "96plate" and "96plateloc" to enable merge with adata.obs
list_96well_plates = [str(x).zfill(2) for x in range(1,25)]
list_df_doublet_grades_plate0 = []
for plate0 in list_96well_plates:
    
    df_doublet_grades_plate0 = pd.read_excel(path_data+'20221004_kathuria_SP17_d4_LAIV_Sort_Doublet_Grading.xlsx',sheet_name='Plate_'+plate0)
    df_doublet_grades_plate0['SampleID'] = 'Spleen17'
    df_doublet_grades_plate0['96plate'] = [str(x) for x in df_doublet_grades_plate0['Plate']]
    
    # p has "% 13" function because for this variable, plates after 12 loop back to 1. So 96plate 13 and 25 are both "p01" 
    # will use both "96plate" and "96plateloc" to merge with adata.obs
    
    df_doublet_grades_plate0['96plateloc'] = ['p'+convert_96plateloc_to_96plate(p)+'c'+str(c).zfill(2)+'r'+str(ord(r)-64).zfill(2) for p,c,r in zip(df_doublet_grades_plate0['Plate'],df_doublet_grades_plate0['Column'],df_doublet_grades_plate0['Row'])]
    
    #append
    list_df_doublet_grades_plate0.append(df_doublet_grades_plate0)

#concatenate 
df_doublet_grades = pd.concat(list_df_doublet_grades_plate0, ignore_index=True)

#drop extraneous columns
df_doublet_grades = df_doublet_grades[['SampleID','96plate','96plateloc','Doublet Score']]
df_doublet_grades.head()

In [ ]:
print(len(df_doublet_grades),len(df_doublet_grades[df_doublet_grades['Doublet Score'].notna()]))

In [ ]:
# adata.obs.merge did not work. Method below is brute force

#merge with adata using for loop
count_failqc = 0
count = 0
for i in range(len(df_doublet_grades)):
    sampleid0 = df_doublet_grades.iloc[i]['SampleID']
    plate0 = df_doublet_grades.iloc[i]['96plate']
    plateloc96 = df_doublet_grades.iloc[i]['96plateloc']
    doublet_score0 = df_doublet_grades.iloc[i]['Doublet Score']
    
    #confirm only 1  meets criteria
    if len(adata.obs.loc[(adata.obs['SampleID']==sampleid0) & (adata.obs['96plate']==plate0) & (adata.obs['96plateloc']==plateloc96)])==1:
        adata.obs.loc[(adata.obs['SampleID']==sampleid0) & (adata.obs['96plate']==plate0) & (adata.obs['96plateloc']==plateloc96),'doubletscore'] = doublet_score0
        count+=1
    #if 0 (did not pass qc) cell
    elif len(adata.obs.loc[(adata.obs['SampleID']==sampleid0) & (adata.obs['96plate']==plate0) & (adata.obs['96plateloc']==plateloc96)])==0:
        print(sampleid0,plate0,plateloc96)
        count_failqc+=1
    else:
        print('More than 1 cell meets criteria:',sampleid0,plate0,plateloc96)
    
print(count_failqc,'cells failed QC after RT-PCR.')

In [ ]:
adata.obs.head()

In [ ]:
# save file

adata.write_h5ad(path_data+'Doublets_3363cells_GeneSymbols_preprocessed_2xDoubletCounts_wDoubletScores.h5ad')

# QC Check

In [ ]:
sc.pl.violin(adata, keys=['n_counts', 'n_genes', 'mt_frac', 'ercc_frac'], groupby='sortedtype', rotation=90, use_raw=True)